# KaroSpace Exploration Compare Annotations

This notebook recreates the quick calculation used by `Exploration > Compare > Annotations` for two annotation values. By default it runs on every feature in the selected assay; set `features` to a short list if you want a smaller test run.

In [11]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse, stats


## Plain variables

In [ ]:
h5ad_path = "./data.h5ad"

assay = "rna"  # "rna" or "protein"
protein_matrix_key = "protein"
protein_feature_table_key = "protein_var"
statistics_counts_layer = "counts"  # Use None for adata.X. Default KaroSpace export is "counts".
statistics_normalization = "RC"  # "RC" or "LogNormalize".
statistics_scale_factor = 10000.0  # Used by RC and LogNormalize.
statistics_normalized_layer = None  # Example: "data". If set, use this layer directly.

annotation_col = "leiden_rna"
annotation_a = "0"  # Set None to use the first available category.
annotation_b = "1"  # Set None to use the second available category.
composition_col = annotation_col  # This is the annotation used in the composition bars.

restrict_col = None  # Example: "sample_id". None disables the Restrict within control.
restrict_value = None

features = None  # None = all features. Example: ["ABCA1", "ACSL4", "ADAM28"]

n_cells_downsample = None  # None = all cells. Match the HTML export downsample if one was used.
random_seed = 7

top_n_per_direction = 5
min_detected_pct = 0  # Same idea as the panel slider. 0 disables this filter.


## Load data and select cells

In [13]:
adata = ad.read_h5ad(h5ad_path)

if assay == "rna":
    assay_matrix = adata.X
    assay_source = "adata.X"
    assay_layers = adata.layers
    feature_names = [str(x) for x in adata.var_names]
elif assay == "protein":
    assay_matrix = adata.obsm[protein_matrix_key]
    assay_source = f"adata.obsm[{protein_matrix_key!r}]"
    assay_layers = {protein_matrix_key: assay_matrix}
    if f"{protein_matrix_key}_arcsinh" in adata.obsm:
        assay_layers[f"{protein_matrix_key}_arcsinh"] = adata.obsm[f"{protein_matrix_key}_arcsinh"]
    protein_var = adata.uns[protein_feature_table_key]
    feature_names = [str(x) for x in protein_var.iloc[:, 0].to_numpy()]
else:
    raise ValueError('assay must be "rna" or "protein"')

def dense(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)

def matrix_for_layer(layer_name):
    if layer_name is None or layer_name == "X":
        return assay_matrix, assay_source
    if layer_name in assay_layers:
        return assay_layers[layer_name], f"{assay} matrix {layer_name!r}"
    if layer_name in adata.layers:
        return adata.layers[layer_name], f"adata.layers[{layer_name!r}]"
    if layer_name in adata.obsm:
        return adata.obsm[layer_name], f"adata.obsm[{layer_name!r}]"
    return assay_matrix, f"{assay_source} (matrix {layer_name!r} not found)"

def library_normalize(matrix, scale_factor):
    if sparse.issparse(matrix):
        out = matrix.astype(float, copy=True).tocsr()
        out.data[~np.isfinite(out.data)] = 0.0
        totals = np.asarray(out.sum(axis=1)).ravel()
        scale = np.zeros_like(totals, dtype=float)
        valid = np.isfinite(totals) & (totals > 0)
        scale[valid] = float(scale_factor) / totals[valid]
        out = sparse.diags(scale).dot(out).tocsr()
        out.data[~np.isfinite(out.data)] = 0.0
        out.eliminate_zeros()
        return out
    out = np.array(matrix, dtype=float, copy=True)
    out[~np.isfinite(out)] = 0.0
    totals = out.sum(axis=1)
    scale = np.zeros_like(totals, dtype=float)
    valid = np.isfinite(totals) & (totals > 0)
    scale[valid] = float(scale_factor) / totals[valid]
    out *= scale[:, None]
    out[~np.isfinite(out)] = 0.0
    return out

def distribution_matrix():
    if statistics_normalized_layer is not None:
        matrix, source = matrix_for_layer(statistics_normalized_layer)
        return matrix, f"{source} directly"
    matrix, source = matrix_for_layer(statistics_counts_layer)
    mode = str(statistics_normalization).strip().lower()
    if mode in {"rc", "relative_counts", "relative-counts", "relativecounts"}:
        return library_normalize(matrix, statistics_scale_factor), f"RC {source}, scale_factor={statistics_scale_factor:g}, no log1p"
    if mode in {"lognormalize", "log_normalize", "log-normalize", "lognorm"}:
        normalized = library_normalize(matrix, statistics_scale_factor)
        if sparse.issparse(normalized):
            normalized = normalized.tocsr(copy=True)
            normalized.data = np.log1p(normalized.data)
            normalized.eliminate_zeros()
        else:
            normalized = np.log1p(normalized)
        return normalized, f"LogNormalize {source}, target_sum={statistics_scale_factor:g}, log1p"
    raise ValueError('statistics_normalization must be RC or LogNormalize')

matrix, matrix_source = distribution_matrix()
feature_pos = {name: i for i, name in enumerate(feature_names)}
if features is None:
    features = list(feature_names)
else:
    features = [str(f) for f in features if str(f) in feature_pos]
    if not features:
        raise ValueError("No requested features were found in the selected assay")

if annotation_col not in adata.obs:
    raise ValueError(f"annotation_col {annotation_col!r} is not in adata.obs")
if composition_col not in adata.obs:
    raise ValueError(f"composition_col {composition_col!r} is not in adata.obs")
if restrict_col is not None and restrict_col not in adata.obs:
    raise ValueError(f"restrict_col {restrict_col!r} is not in adata.obs")

valid = adata.obs[annotation_col].notna().to_numpy().copy()
valid &= adata.obs[composition_col].notna().to_numpy()
if restrict_col is not None:
    valid &= adata.obs[restrict_col].notna().to_numpy()
    if restrict_value is not None:
        valid &= adata.obs[restrict_col].astype(str).to_numpy() == str(restrict_value)

cell_idx = np.flatnonzero(valid)
if n_cells_downsample is not None and len(cell_idx) > int(n_cells_downsample):
    rng = np.random.default_rng(random_seed)
    cell_idx = np.sort(rng.choice(cell_idx, size=int(n_cells_downsample), replace=False))

obs = adata.obs.iloc[cell_idx].copy()
labels = obs[annotation_col].astype(str)
categories = [str(x) for x in pd.unique(labels)]
if annotation_a is None:
    annotation_a = categories[0]
if annotation_b is None:
    annotation_b = next(x for x in categories if x != str(annotation_a))
annotation_a = str(annotation_a)
annotation_b = str(annotation_b)

mask_a = labels.to_numpy() == annotation_a
mask_b = labels.to_numpy() == annotation_b
if mask_a.sum() < 2 or mask_b.sum() < 2:
    raise ValueError(f"Each annotation needs at least two cells. A={mask_a.sum()}, B={mask_b.sum()}")

selected_sparse_matrix = matrix[cell_idx, :].tocsc() if sparse.issparse(matrix) else None

def feature_values_for_cells(feature):
    col = feature_pos[str(feature)]
    if selected_sparse_matrix is not None:
        x = dense(selected_sparse_matrix[:, col]).astype(float, copy=False).ravel()
    else:
        x = np.asarray(matrix[cell_idx, col], dtype=float).ravel()
    x[~np.isfinite(x)] = 0.0
    return x

print(f"Matrix source: {matrix_source}")
print(f"Cells used: {len(cell_idx):,}")
print(f"Annotation A: {annotation_a} ({mask_a.sum():,} cells)")
print(f"Annotation B: {annotation_b} ({mask_b.sum():,} cells)")
print(f"Features tested: {len(features):,}")
print(f"First features: {features[:5]}")


Matrix source: RC adata.X (matrix 'counts' not found), scale_factor=10000, no log1p
Cells used: 46,003
Annotation A: 0 (4,444 cells)
Annotation B: 1 (6,233 cells)
Features tested: 5,001
First features: ['A2ML1', 'AAMP', 'AAR2', 'AARSD1', 'ABAT']


## Cell composition by annotation

The HTML panel builds composition bars from the cells in Annotation A and Annotation B, grouped by the active composition annotation.

In [14]:
def composition_table(mask, name):
    s = obs.loc[mask, composition_col].astype(str)
    counts = s.value_counts().sort_index()
    total = int(mask.sum())
    return pd.DataFrame({
        "Compared annotation": name,
        "Composition category": counts.index,
        "Cells": counts.to_numpy(dtype=int),
        "% Cells": 100.0 * counts.to_numpy(dtype=float) / max(total, 1),
    })

composition = pd.concat([
    composition_table(mask_a, annotation_a),
    composition_table(mask_b, annotation_b),
], ignore_index=True)

counts_summary = pd.DataFrame([
    {"Annotation": annotation_a, "Cells": int(mask_a.sum())},
    {"Annotation": annotation_b, "Cells": int(mask_b.sum())},
])

display(counts_summary)
display(composition)


,Annotation,Cells
0,0,4444
1,1,6233


,Compared annotation,Composition category,Cells,% Cells
0,0,0,4444,100.0
1,1,1,6233,100.0


## Welch feature comparison

This reproduces the quick marker calculation: for each loaded feature, calculate mean, detected fraction, log2 fold-change, Welch t score, Welch degrees of freedom, and two-sided p-value.

In [15]:
def feature_stats(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return {
        "n": int(x.size),
        "sum": float(x.sum()),
        "sumSq": float(np.square(x).sum()),
        "nnz": int(np.count_nonzero(x > 0)),
    }

def welch_row(feature, a, b):
    sa = feature_stats(a)
    sb = feature_stats(b)
    n_a = sa["n"]
    n_b = sb["n"]
    if n_a < 2 or n_b < 2:
        return None
    mean_a = sa["sum"] / n_a
    mean_b = sb["sum"] / n_b
    var_a = max(0.0, (sa["sumSq"] - n_a * mean_a * mean_a) / (n_a - 1))
    var_b = max(0.0, (sb["sumSq"] - n_b * mean_b * mean_b) / (n_b - 1))
    se2 = var_a / n_a + var_b / n_b
    t = (mean_a - mean_b) / np.sqrt(se2) if se2 > 0 else 0.0
    df_denom = (var_a / n_a) ** 2 / (n_a - 1) + (var_b / n_b) ** 2 / (n_b - 1)
    df = (se2 ** 2 / df_denom) if df_denom > 0 else (n_a + n_b - 2)
    tiny = np.nextafter(0, 1)
    log2fc = np.log2(max(mean_a, 0) + tiny) - np.log2(max(mean_b, 0) + tiny)
    return {
        "Feature": feature,
        "Mean A": mean_a,
        "Mean B": mean_b,
        "% detected A": sa["nnz"] / n_a,
        "% detected B": sb["nnz"] / n_b,
        "log2FC": log2fc,
        "Score": t,
        "t": t,
        "df": df,
        "p": 2.0 * stats.t.sf(abs(t), df) if np.isfinite(df) and df > 0 else 1.0,
        "nA": n_a,
        "nB": n_b,
    }

rows = []
for feature in features:
    feature_values = feature_values_for_cells(feature)
    row = welch_row(feature, feature_values[mask_a], feature_values[mask_b])
    if row is not None:
        rows.append(row)

welch_all = pd.DataFrame(rows).sort_values(
    ["p", "t", "Feature"],
    ascending=[True, False, True],
).reset_index(drop=True)

threshold = max(0, min(100, float(min_detected_pct))) / 100.0
eligible = welch_all[(welch_all["% detected A"] >= threshold) | (welch_all["% detected B"] >= threshold)].copy()
positive = eligible[eligible["Score"] >= 0].sort_values(["Score", "p"], ascending=[False, True]).head(int(top_n_per_direction))
negative = eligible[eligible["Score"] < 0].sort_values(["Score", "p"], ascending=[True, True]).head(int(top_n_per_direction))
displayed = pd.concat([positive, negative], ignore_index=True)

display(welch_all)
print("Displayed top rows")
display(displayed)


,Feature,Mean A,Mean B,% detected A,% detected B,log2FC,Score,t,df,p,nA,nB
0,APOB,254.506029,5.765651,0.980873,0.078293,5.464072,106.671401,106.671401,4776.738940,0.000000,4444,6233
1,IGF2,241.414841,10.650117,0.996400,0.131237,4.502573,99.227713,99.227713,5267.690355,0.000000,4444,6233
2,SERPINA5,29.108174,1.806975,0.829883,0.040430,4.009776,66.449099,66.449099,6016.644062,0.000000,4444,6233
3,EPCAM,30.108415,2.379719,0.842484,0.045564,3.661304,64.665238,64.665238,6204.067125,0.000000,4444,6233
4,HSPD1,47.928771,9.112256,0.916067,0.174555,2.395012,59.187390,59.187390,9605.855111,0.000000,4444,6233
...,...,...,...,...,...,...,...,...,...,...,...,...
4996,FGF19,0.074155,0.073962,0.006526,0.001765,0.003761,0.005726,0.005726,10670.239370,0.995432,4444,6233
4997,CCDC91,2.262776,2.263741,0.205671,0.041874,-0.000615,-0.003735,-0.003735,7831.927870,0.997020,4444,6233
4998,NLRP11,0.093769,0.093939,0.011701,0.001765,-0.002613,-0.003726,-0.003726,7607.492317,0.997027,4444,6233
4999,MFSD5,0.666362,0.666752,0.070432,0.017327,-0.000845,-0.003358,-0.003358,8392.596381,0.997321,4444,6233


Displayed top rows


,Feature,Mean A,Mean B,% detected A,% detected B,log2FC,Score,t,df,p,nA,nB
0,APOB,254.506029,5.765651,0.980873,0.078293,5.464072,106.671401,106.671401,4776.738940,0.000000e+00,4444,6233
1,IGF2,241.414841,10.650117,0.996400,0.131237,4.502573,99.227713,99.227713,5267.690355,0.000000e+00,4444,6233
2,SERPINA5,29.108174,1.806975,0.829883,0.040430,4.009776,66.449099,66.449099,6016.644062,0.000000e+00,4444,6233
3,EPCAM,30.108415,2.379719,0.842484,0.045564,3.661304,64.665238,64.665238,6204.067125,0.000000e+00,4444,6233
4,HSPD1,47.928771,9.112256,0.916067,0.174555,2.395012,59.187390,59.187390,9605.855111,0.000000e+00,4444,6233
5,COL4A1,0.948660,36.417548,0.067957,0.285416,-5.262599,-32.647032,-32.647032,6305.819348,3.815702e-216,4444,6233
6,THY1,0.286657,26.003379,0.020252,0.284454,-6.503231,-32.436878,-32.436878,6291.699693,1.451468e-213,4444,6233
7,COL4A2,0.599847,22.157863,0.046805,0.269694,-5.207080,-32.235309,-32.235309,6355.538047,2.744882e-211,4444,6233
8,COL5A1,0.184737,18.581459,0.014176,0.214824,-6.652249,-29.432528,-29.432528,6274.929005,1.789337e-178,4444,6233
9,TNC,0.125039,14.127920,0.012601,0.212257,-6.820025,-28.675968,-28.675968,6253.095891,5.147435e-170,4444,6233
